# Station Training Baseline — RKPK: Busan Gimhae International Airport

This is the canonical single-model station workflow. It trains one XGBoost point regressor, a conditional Gaussian residual probability baseline, and four ordinal research candidates. Blended, shared-slope, and pure ordinal form the canonical two-of-three ensemble; the native ordinal reference is non-voting. Validation is chronological, and production artifacts are separately labeled candidates.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('weather-research project root not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.calibration.station_baseline import (
    ARCHITECTURE_VERSION,
    load_station_config,
    run_station_baseline,
)


## Station and chronology contract


In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'notebooks/station_training_baseline/configs/RKPK.json'
station_config = load_station_config(CONFIG_PATH)
{
    'architecture': ARCHITECTURE_VERSION,
    'station_id': station_config['station_id'],
    'timezone': station_config['timezone'],
    'providers': station_config['providers'],
    'point_model': 'xgboost',
    'optuna_trials': station_config['optuna_trials'],
    'optuna_startup_trials': station_config['optuna_startup_trials'],
    'probability_benchmark': 'conditional_gaussian_residual',
    'ordinal_candidates': [
        'native_ordinal_reference',
        'blended_ordinal',
        'shared_slope_ordinal',
        'pure_ordinal',
    ],
    'ordinal_ensemble': {
        'voting_members': ['blended_ordinal', 'shared_slope_ordinal', 'pure_ordinal'],
        'required_votes': 2,
        'aggregation': 'median_selected_bucket',
    },
}


## Train, validate, compare, and export

This call writes frozen validation artifacts, forward/holdout comparison reports, and separate live-production candidate artifacts. Persistent Optuna storage resumes until the configured total of 100 XGBoost trials is complete.


In [ ]:
run = run_station_baseline(
    CONFIG_PATH,
    project_root=PROJECT_ROOT,
    export_production=True,  # exports an unapproved production candidate
)
run.point_scoreboard


## Gaussian, ordinal candidates, and ensemble comparison


In [ ]:
run.probability_comparison


## Exported validation and production-candidate artifacts


In [ ]:
{name: str(path) for name, path in run.artifact_paths.items()}


## Reports


In [ ]:
{name: str(path) for name, path in run.report_paths.items()}
